![FLIP Banner](../../Assets/images/flip-banner.png)

# FLIP: Agentic AI in Practice
**(Module 04: LangChain Programming)**

---

- Materials in this module have been developed to support practical learning in generative AI and agentic AI systems.
- Teaching content is licensed under CC BY 4.0 and code under MIT; see [LICENSING.md](../../LICENSING.md) for scope and exclusions.
- If you find any issue or bug in this document, please submit an issue at [tulip-lab/agentic-ai](https://github.com/tulip-lab/agentic-ai/issues).

Prepared by :tulip: **[TULIP Lab](https://www.tulip.academy), Australia**

---

## Session 4D: Lead Research and Writing Agent

<div align="center">

<table>
<thead>
<tr><th><strong>Item</strong></th><th><strong>Description</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Estimated time</td><td>2 hours</td></tr>
<tr><td align="left">Environment</td><td>Google Colab or local Jupyter</td></tr>
<tr><td align="left">Mandatory part</td><td>Local mock research-and-writing agent using approved public snippets</td></tr>
<tr><td align="left">Optional part</td><td>Real model drafting if packages and API key are available</td></tr>
<tr><td align="left">Main output</td><td>A short evidence-grounded lead-research summary and outreach draft</td></tr>
</tbody>
</table>

</div>

---

**Table of Contents**

1. [Overview and Learning Goals](#m04d-overview)
2. [Setup and Background](#m04d-setup)
3. [Public Evidence Store](#m04d-evidence)
4. [Mandatory Local Research and Writing Agent](#m04d-agent)
5. [Optional Real Model Drafting](#m04d-real)
6. [Testing and Analysis](#m04d-testing)
7. [Student Tasks](#m04d-student-tasks)
8. [Submission and Reflection](#m04d-submission)

---

<a id="m04d-overview"></a>

### 1. Overview and Learning Goals

M04D combines the main ideas from M04A, M04B and M04C into a small end-to-end agentic workflow. M04A introduced prompt-model-parser chains. M04B introduced controlled tools. M04C introduced safe local storage and state-changing actions. M04D uses these ideas to build a **lead research and writing agent**.

In this notebook, “lead research” does not mean scraping the web or contacting real people. It means analysing a small approved set of public snippets and producing a structured research summary and a draft outreach message. The agent will not send email. It will only draft text.

```mermaid
flowchart LR
    A[Research request] --> B[Validate topic]
    B --> C[Search approved public snippets]
    C --> D[Extract evidence]
    D --> E[Write structured summary]
    E --> F[Draft outreach message]
    F --> G[Review checklist]
```

This design is intentionally conservative. Many real lead-generation systems connect to search engines, CRMs, email systems, calendars and analytics. Those are higher-risk external actions. In this teaching lab, the workflow stays local and controlled so you can focus on architecture, evidence use and safe drafting.

By the end of this session, you should be able to build a small research-and-writing pipeline, use only approved evidence, produce structured output, avoid unsupported claims, test boundary cases, and explain how this prepares for more advanced agent workflows in M05 and M08.

<a id="m04d-setup"></a>

### 2. Setup and Background

#### 2.1 What is a research-and-writing agent?

A research-and-writing agent is a workflow that collects relevant information, organises it, and drafts text for a human to review. It should not be treated as an autonomous salesperson or decision-maker. The final message should be checked by a human before use.

In this notebook, the agent works with a small in-memory evidence store. It can:

```text
1. search approved public snippets,
2. select relevant evidence,
3. build a structured summary,
4. draft a short outreach message,
5. flag limitations and missing evidence.
```

It cannot:

```text
1. browse the live web,
2. contact real people,
3. send email,
4. access private databases,
5. invent unsupported claims,
6. use hidden instructor materials.
```

#### 2.2 Why this session belongs after M04C

M04C showed that state-changing actions require validation. M04D adds another important idea: writing agents require evidence control. A generated outreach draft can be persuasive even when unsupported. Therefore, the workflow must make evidence and limitations visible.

```mermaid
flowchart LR
    A[Evidence] --> B[Summary]
    B --> C[Draft]
    C --> D[Human review]
    E[No evidence] --> F[State limitation]
    F --> D
```

In [ ]:
import json
import re
from dataclasses import dataclass
from typing import Any, Dict, List, Optional

print("M04D setup complete.")

<a id="m04d-evidence"></a>

### 3. Public Evidence Store

The evidence store below is a local teaching dataset. It contains short public-style snippets about possible organisations and their AI-related interests. In a real system, evidence might come from public websites, public reports or approved CRM notes. In this notebook, we use a controlled local dataset to avoid privacy, scraping and external-action risks.

Each evidence item has:

```text
lead_id: identifier
organisation: organisation name
sector: broad sector
snippet: public-style evidence text
tags: searchable topic tags
source: source description or URL placeholder
```

In [ ]:
PUBLIC_EVIDENCE = [
    {
        "lead_id": "L001",
        "organisation": "Northbank Health Analytics",
        "sector": "health",
        "snippet": "Northbank Health Analytics publishes public reports on hospital demand forecasting and responsible AI governance.",
        "tags": ["health", "forecasting", "responsible_ai", "governance"],
        "source": "public report summary"
    },
    {
        "lead_id": "L002",
        "organisation": "Harbour Retail Group",
        "sector": "retail",
        "snippet": "Harbour Retail Group has discussed customer-service automation, product recommendation and privacy-preserving analytics in public innovation updates.",
        "tags": ["retail", "recommendation", "customer_service", "privacy"],
        "source": "public innovation update"
    },
    {
        "lead_id": "L003",
        "organisation": "GreenField Smart Farming",
        "sector": "agriculture",
        "snippet": "GreenField Smart Farming explores sensor-based crop monitoring, weather-aware decision support and AI-assisted irrigation planning.",
        "tags": ["agriculture", "smart_farming", "iot", "decision_support"],
        "source": "public project description"
    },
    {
        "lead_id": "L004",
        "organisation": "MetroCyber Training Institute",
        "sector": "education",
        "snippet": "MetroCyber Training Institute offers public short courses on cybersecurity awareness, AI safety and digital-skills training.",
        "tags": ["education", "cybersecurity", "ai_safety", "training"],
        "source": "public course catalogue"
    },
    {
        "lead_id": "L005",
        "organisation": "Civic Transport Lab",
        "sector": "transport",
        "snippet": "Civic Transport Lab publishes open material on traffic prediction, route optimisation and data-driven transport planning.",
        "tags": ["transport", "prediction", "optimisation", "planning"],
        "source": "public research page"
    },
]

len(PUBLIC_EVIDENCE), PUBLIC_EVIDENCE[0]

The evidence store is intentionally small. This makes it possible to inspect exactly why the agent produced a summary. A real agent should preserve the same habit: show which evidence was used and avoid unsupported claims.

In [ ]:
def normalise_text(text: str) -> str:
    return re.sub(r"\s+", " ", text.lower()).strip()


def search_evidence(query: str, evidence_store: List[Dict[str, Any]], top_k: int = 3) -> Dict[str, Any]:
    """Search the approved evidence store using simple keyword overlap."""

    if not isinstance(query, str) or not query.strip():
        return {"ok": False, "error": "query must be a non-empty string.", "result": None}

    if not isinstance(top_k, int) or top_k <= 0:
        return {"ok": False, "error": "top_k must be a positive integer.", "result": None}

    query_terms = set(re.findall(r"[a-zA-Z_]+", normalise_text(query)))
    scored = []

    for item in evidence_store:
        item_text = " ".join([
            item.get("organisation", ""),
            item.get("sector", ""),
            item.get("snippet", ""),
            " ".join(item.get("tags", [])),
        ])
        item_terms = set(re.findall(r"[a-zA-Z_]+", normalise_text(item_text)))
        score = len(query_terms.intersection(item_terms))
        if score > 0:
            scored.append((score, item))

    scored.sort(key=lambda x: x[0], reverse=True)
    results = [item for _, item in scored[:top_k]]

    return {"ok": True, "error": None, "result": results}


search_evidence("AI safety training education", PUBLIC_EVIDENCE)

This search function is simple on purpose. It is not a vector database and not a live search engine. Its role is to demonstrate the evidence-selection step. Later RAG sessions will use stronger retrieval methods.

<a id="m04d-agent"></a>

### 4. Mandatory Local Research and Writing Agent

The local agent has three stages:

```text
1. retrieve relevant approved evidence,
2. build a structured summary,
3. draft an outreach message for human review.
```

It refuses unsafe requests such as contacting real people, sending email, scraping private websites or accessing credentials.

In [ ]:
def check_request_safety(request: str) -> Dict[str, Any]:
    if not isinstance(request, str) or not request.strip():
        return {"ok": False, "error": "request must be a non-empty string.", "result": None}

    lower = request.lower()
    unsafe_keywords = [
        "send email", "send message", "contact them", "scrape", "private database",
        "password", "api key", "credential", "hidden solution", "student record",
        "personal phone", "personal email"
    ]

    if any(keyword in lower for keyword in unsafe_keywords):
        return {
            "ok": True,
            "error": None,
            "result": {
                "safe": False,
                "reason": "The request asks for external contact, private data, credentials, scraping, or hidden material."
            }
        }

    return {"ok": True, "error": None, "result": {"safe": True, "reason": "Allowed local research-and-drafting request."}}


print(check_request_safety("Draft an outreach message for AI safety training."))
print(check_request_safety("Send email to all leads."))

In [ ]:
def build_research_summary(query: str, evidence_items: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Build a structured research summary from retrieved evidence."""

    if not evidence_items:
        return {
            "ok": True,
            "error": None,
            "result": {
                "query": query,
                "matched_leads": [],
                "summary": "No sufficiently relevant approved evidence was found.",
                "limitations": ["The approved evidence store may not contain this topic."],
            }
        }

    matched = []
    for item in evidence_items:
        matched.append({
            "lead_id": item["lead_id"],
            "organisation": item["organisation"],
            "sector": item["sector"],
            "evidence": item["snippet"],
            "source": item["source"],
        })

    themes = sorted(set(tag for item in evidence_items for tag in item.get("tags", [])))

    summary = (
        f"Found {len(evidence_items)} relevant lead(s) for the query. "
        f"Common evidence themes include: {', '.join(themes[:8])}."
    )

    return {
        "ok": True,
        "error": None,
        "result": {
            "query": query,
            "matched_leads": matched,
            "summary": summary,
            "limitations": [
                "This summary uses only the approved local evidence store.",
                "It should not be treated as live web research.",
                "A human should verify all claims before use."
            ],
        }
    }


evidence = search_evidence("AI safety training education", PUBLIC_EVIDENCE)["result"]
summary = build_research_summary("AI safety training education", evidence)
summary

In [ ]:
def draft_outreach_message(summary_result: Dict[str, Any], audience: str = "potential collaborator") -> Dict[str, Any]:
    """Draft a short outreach message based only on structured summary evidence."""

    if not isinstance(summary_result, dict) or "matched_leads" not in summary_result:
        return {"ok": False, "error": "summary_result is not valid.", "result": None}

    matched = summary_result["matched_leads"]

    if not matched:
        return {
            "ok": True,
            "error": None,
            "result": {
                "draft": (
                    "I could not find enough approved evidence to draft a specific outreach message. "
                    "Please refine the topic or provide approved public context."
                ),
                "requires_review": True,
                "evidence_used": [],
            }
        }

    first = matched[0]
    draft = (
        f"Dear {first['organisation']} team,\n\n"
        f"I am reaching out regarding possible collaboration around {summary_result['query']}. "
        f"I noticed public information indicating your interest in {first['evidence']} "
        f"This appears aligned with practical work in agentic AI, responsible AI and applied data-driven systems.\n\n"
        f"If relevant, I would be pleased to discuss whether there is scope for a short exploratory conversation.\n\n"
        f"Kind regards,"
    )

    return {
        "ok": True,
        "error": None,
        "result": {
            "draft": draft,
            "requires_review": True,
            "evidence_used": [first],
        }
    }


draft = draft_outreach_message(summary["result"])
print(draft["result"]["draft"])

The draft uses only the selected evidence. It does not invent personal contact details, does not claim private access, and does not send anything. It also marks the draft as requiring human review.

In [ ]:
class LeadResearchWritingAgent:
    """Controlled local research-and-writing agent using approved evidence only."""

    def __init__(self, evidence_store: List[Dict[str, Any]]):
        self.evidence_store = evidence_store

    def invoke(self, request: str, top_k: int = 3) -> Dict[str, Any]:
        safety = check_request_safety(request)
        if not safety["ok"]:
            return safety

        if not safety["result"]["safe"]:
            return {
                "ok": True,
                "error": None,
                "result": {
                    "action": "refuse",
                    "answer": safety["result"]["reason"],
                }
            }

        retrieved = search_evidence(request, self.evidence_store, top_k=top_k)
        if not retrieved["ok"]:
            return retrieved

        summary = build_research_summary(request, retrieved["result"])
        if not summary["ok"]:
            return summary

        draft = draft_outreach_message(summary["result"])
        if not draft["ok"]:
            return draft

        return {
            "ok": True,
            "error": None,
            "result": {
                "action": "research_and_draft",
                "summary": summary["result"],
                "draft": draft["result"],
            }
        }


agent = LeadResearchWritingAgent(PUBLIC_EVIDENCE)
agent_result = agent.invoke("AI safety training education")
agent_result

In [ ]:
def display_research_agent_result(agent_result: Dict[str, Any]) -> None:
    if not agent_result.get("ok"):
        print("ERROR:", agent_result.get("error"))
        return

    result = agent_result["result"]
    print("Action:", result.get("action"))

    if result.get("action") == "refuse":
        print("Answer:", result.get("answer"))
        return

    summary = result["summary"]
    print("\nSummary:")
    print(summary["summary"])

    print("\nMatched leads:")
    for lead in summary["matched_leads"]:
        print(f"- {lead['lead_id']} | {lead['organisation']} | {lead['sector']}")
        print(f"  Evidence: {lead['evidence']}")

    print("\nLimitations:")
    for limitation in summary["limitations"]:
        print("-", limitation)

    print("\nDraft:")
    print(result["draft"]["draft"])


display_research_agent_result(agent_result)

The output is intentionally inspectable. A student should be able to see the evidence, the summary, the limitations and the draft. If any part is unsupported, the issue should be visible.

<a id="m04d-real"></a>

### 5. Optional Real Model Drafting

This optional section shows how a real model could be used to rewrite a draft. It is not required for the core task. Use it only if you have a valid API key, internet access and the relevant packages installed.

The real model should not be allowed to add unsupported claims. It should rewrite or polish only from the structured evidence.

In [ ]:
# Optional installation cell.
# Uncomment only when package installation is allowed.

# !pip install -q langchain langchain-core langchain-openai

In [ ]:
def optional_real_rewrite(draft_text: str, evidence_used: List[Dict[str, Any]]) -> Dict[str, Any]:
    import os

    if not os.environ.get("OPENAI_API_KEY"):
        return {
            "ok": False,
            "error": "OPENAI_API_KEY is not set. Skip this optional section or set the key securely.",
            "result": None,
        }

    try:
        from langchain_openai import ChatOpenAI
        from langchain_core.prompts import ChatPromptTemplate
        from langchain_core.output_parsers import StrOutputParser
    except ImportError as exc:
        return {
            "ok": False,
            "error": f"Required LangChain packages are not installed: {exc}",
            "result": None,
        }

    evidence_text = json.dumps(evidence_used, indent=2)

    prompt = ChatPromptTemplate.from_messages([
        ("system", "Rewrite the draft for clarity and professionalism. Use only the supplied evidence. Do not add unsupported claims. Do not send anything."),
        ("human", "Evidence:\n{evidence}\n\nDraft:\n{draft}")
    ])

    model = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
    parser = StrOutputParser()
    chain = prompt | model | parser

    rewritten = chain.invoke({"evidence": evidence_text, "draft": draft_text})

    return {"ok": True, "error": None, "result": rewritten}


optional_rewrite = optional_real_rewrite(
    agent_result["result"]["draft"]["draft"],
    agent_result["result"]["draft"]["evidence_used"],
)
optional_rewrite

If the optional real rewrite cannot run, record it as skipped. The mandatory learning outcome is the local evidence-grounded research-and-writing agent.

<a id="m04d-testing"></a>

### 6. Testing and Analysis

A research-and-writing agent must be tested for normal evidence use, weak/no evidence, unsafe requests and draft grounding.

In [ ]:
test_agent = LeadResearchWritingAgent(PUBLIC_EVIDENCE)

# Normal: education and AI safety.
normal = test_agent.invoke("AI safety training education")
assert normal["ok"] is True
assert normal["result"]["action"] == "research_and_draft"
assert len(normal["result"]["summary"]["matched_leads"]) >= 1
assert normal["result"]["draft"]["requires_review"] is True

# Normal: smart farming.
farming = test_agent.invoke("smart farming irrigation")
assert farming["ok"] is True
assert farming["result"]["action"] == "research_and_draft"
assert any("GreenField" in lead["organisation"] for lead in farming["result"]["summary"]["matched_leads"])

# Weak evidence: unrelated topic.
weak = test_agent.invoke("quantum poetry festival")
assert weak["ok"] is True
assert weak["result"]["action"] == "research_and_draft"
assert weak["result"]["summary"]["matched_leads"] == []
assert "not find enough" in weak["result"]["draft"]["draft"].lower()

# Boundary: send email is refused.
send_email = test_agent.invoke("send email to all leads about AI training")
assert send_email["ok"] is True
assert send_email["result"]["action"] == "refuse"

# Boundary: private database is refused.
private_db = test_agent.invoke("use the private database to find personal emails")
assert private_db["ok"] is True
assert private_db["result"]["action"] == "refuse"

# Failure: empty request.
empty = test_agent.invoke("")
assert empty["ok"] is False

print("All M04D mandatory research-agent tests passed.")

In [ ]:
# Inspect representative outcomes.

for request in [
    "AI safety training education",
    "smart farming irrigation",
    "quantum poetry festival",
    "send email to all leads",
]:
    print("\n==============================")
    print("REQUEST:", request)
    print("==============================")
    display_research_agent_result(test_agent.invoke(request))

The tests show four important behaviours: grounded draft, alternative topic match, insufficient-evidence handling and refusal. These behaviours are more important than making the writing sound impressive.

<a id="m04d-student-tasks"></a>

### 7. Student Tasks

Complete the tasks below. The mandatory local research-and-writing agent must run without external API calls.

<div align="center">

<table>
<thead>
<tr><th><strong>Task</strong></th><th><strong>What to do</strong></th><th><strong>Detailed instructions</strong></th><th><strong>Evidence to submit</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Task 1: Run baseline tests</td><td>Run the mandatory agent.</td><td>Run all cells through the testing section.</td><td>Output showing <code>All M04D mandatory research-agent tests passed.</code></td></tr>
<tr><td align="left">Task 2: Add one evidence item</td><td>Add a new public-style lead.</td><td>Add one new dictionary to <code>PUBLIC_EVIDENCE</code>. It must include <code>lead_id</code>, <code>organisation</code>, <code>sector</code>, <code>snippet</code>, <code>tags</code> and <code>source</code>.</td><td>Updated evidence item.</td></tr>
<tr><td align="left">Task 3: Query your new lead</td><td>Run the agent on a query matching your new evidence.</td><td>Use a query that retrieves your new lead in the top results.</td><td>Displayed summary and draft.</td></tr>
<tr><td align="left">Task 4: Add tests</td><td>Add at least three tests.</td><td>Include one normal test for your new lead, one weak/no-evidence test and one unsafe-request refusal test.</td><td>Test cell with <code>assert</code> statements.</td></tr>
<tr><td align="left">Task 5: Check grounding</td><td>Explain evidence use.</td><td>Identify which evidence snippet was used in the draft and whether the draft added unsupported claims.</td><td>Short evidence-grounding paragraph.</td></tr>
<tr><td align="left">Task 6: Optional real rewrite</td><td>Run or skip the optional real rewrite.</td><td>If you have API access, run it safely. If not, write <code>Skipped: no API key available</code>.</td><td>Real rewrite output or skipped note.</td></tr>
<tr><td align="left">Task 7: Reflection</td><td>Write a short explanation.</td><td>Explain why writing agents should show evidence and require human review.</td><td>150–250 words.</td></tr>
</tbody>
</table>

</div>

In [ ]:
# Student task starter.
# Add one new approved evidence item.

# Example structure:
# new_item = {
#     "lead_id": "L006",
#     "organisation": "Example Public Organisation",
#     "sector": "education",
#     "snippet": "Example Public Organisation publishes public material on AI training and digital transformation.",
#     "tags": ["education", "ai_training", "digital_transformation"],
#     "source": "public website summary"
# }
#
# PUBLIC_EVIDENCE.append(new_item)
#
# Then create a new LeadResearchWritingAgent(PUBLIC_EVIDENCE)
# and test a query that should retrieve your new item.

<a id="m04d-submission"></a>

### 8. Submission and Reflection

Submit the completed notebook with:

```text
1. Mandatory baseline test output.
2. Your added evidence item.
3. Agent output for a query matching your new evidence.
4. At least three added tests using assert statements.
5. Evidence-grounding paragraph.
6. Optional real rewrite output or skipped note.
7. 150–250 word reflection.
```

Reflection questions:

1. Why should a research-and-writing agent show evidence?
2. What is the risk of allowing the agent to invent lead information?
3. Why does the notebook draft messages but not send them?
4. What is the difference between weak evidence and unsafe requests?
5. How does this prepare for M05 RAG, M06 safety and M08 productised agents?

#### Further Readings

- LangChain tools documentation: <https://python.langchain.com/docs/concepts/tools/>
- LangChain agents overview: <https://python.langchain.com/docs/concepts/agents/>
- LangChain output parsers: <https://python.langchain.com/docs/concepts/output_parsers/>
- LangGraph documentation: <https://langchain-ai.github.io/langgraph/>
- Public data repository for this unit: <https://github.com/tulip-lab/open-data>